# Scattering off a dielectric cylinder (Mie)

A plane wave with in-plane electric field hits an infinite dielectric cylinder of radius
$R = 0.25$ m and refractive index $n = 1.5$ at $k_0 R = 1.5$. The scattering width from the
finite-element solution is compared with the Mie series, and the field is plotted.
Everything in SI units; the time dependence is $e^{-i\omega t}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hpfem
from hpfem import interop

hpfem.set_log_level("warn")
k, radius, index = 6.0, 0.25, 1.5

## Mesh and space

`square_with_disc` resolves the circle with curved cell edges; the inclusion cells carry tag
2, the outer sides the `box_tag`s. A Nédélec space of order 3 on it:

In [ ]:
mesh = hpfem.square_with_disc(4, radius, half_width=1.0, outer=2.0, inclusion_tag=2)
dofs = hpfem.NedelecDofMap2D(mesh, 3)
interop.plot_mesh(mesh)
print(mesh, dofs)

## Problem

Scattered-field formulation: the incident plane wave enters as a volume source on the
scatterer, a PML of one wavelength absorbs the scattered wave, PEC behind it.

In [ ]:
setup = hpfem.ScatteringSetup2D()
setup.omega = k * hpfem.constants.c0
setup.materials.set(2, hpfem.Material.dielectric(index))
setup.incident = hpfem.plane_wave([0.0, 1.0], [k, 0.0])
setup.formulation = hpfem.Formulation.SCATTERED_FIELD
setup.pml = hpfem.PmlBox2D.uniform([-1, -1], [1, 1], 1.0, k, 1.0, hpfem.PmlProfile(2, 1e-10))
setup.pec_tags = [hpfem.box_tag.X_MIN, hpfem.box_tag.X_MAX, hpfem.box_tag.Y_MIN, hpfem.box_tag.Y_MAX]
problem = hpfem.Scattering2D(dofs, setup)
solution = problem.solve()

## Scattering width

From the Poynting flux through the cylinder surface and from the Stratton–Chu far field,
against the Mie series:

In [ ]:
surface = hpfem.Surface2D.around_cells(mesh, 2)
cs = hpfem.cross_sections(problem, solution, surface, incident_amplitude=1.0)
far = hpfem.FarField2D(mesh, surface, hpfem.discrete_field(dofs, solution.unknown),
                       setup.omega, hpfem.Material.vacuum(), order=8)
exact = hpfem.mie_cylinder_scattering_width(k, radius, index)
print(f"sigma_sca (flux)      = {cs.scattering:.6f} m")
print(f"sigma_sca (far field) = {far.scattering_cross_section(1.0):.6f} m")
print(f"sigma_sca (Mie)       = {exact:.6f} m")
print(f"sigma_abs             = {cs.absorption:.2e} m  (lossless: discretisation error)")
interop.plot_far_field(far)

## Field

The scattered field (the unknown of this formulation) and the total field, sampled on the
subdivided mesh. The PML region beyond $|x|, |y| > 1$ shows the decaying wave.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
interop.plot_field(dofs, solution.unknown, ax=axes[0], component="abs", subdivisions=3)
axes[0].set_title("|E_sc|")
locator = hpfem.PointLocator2D(mesh)
x = np.linspace(-1, 1, 161)
line = [problem.total_field(solution, locator, [xi, 0.0]) for xi in x]
axes[1].plot(x, [abs(v[1]) for v in line])
axes[1].set_xlabel("x [m]"); axes[1].set_ylabel("|E_y| on y = 0"); axes[1].set_title("total field")
plt.tight_layout()

## Export

`FieldExporter2D` writes the field on the subdivided mesh for ParaView; `interop.to_meshio`
gives any format meshio writes.

In [ ]:
hpfem.FieldExporter2D(mesh, 3).hcurl("E_sc", dofs, solution.unknown).write("mie_cylinder.vtu")
print(open("mie_cylinder.vtu").read(120))